# El precio de elegir en Biobío
## Territorio, edad, instituciones y aranceles en la Educación Superior · 2021

**Trabajo Práctico — Estadística Descriptiva**  
**Integrantes:** _(completar)_ · **Sección:** _(completar)_ · **Docente:** _(completar)_

### Pregunta guía
**¿Qué diferencias enfrenta una persona al elegir dónde y qué estudiar dentro de la Región del Biobío?**

El trabajo mantiene las preguntas obligatorias de la evaluación, pero las conecta mediante un hilo regional: **territorio, edad, tipo de institución y costo**. Como el análisis es descriptivo, se hablará de patrones y asociaciones observadas, no de causas demostradas.

## 1. Datos y unidades de análisis
La base contiene **106.555 matrículas**. Cada fila representa una matrícula, por lo que una misma oferta puede repetirse muchas veces.

Usaremos dos unidades:
- **Matrículas** para edad, género y tipo de institución.
- **Ofertas académicas únicas** para precios, evitando que una oferta con muchos estudiantes pese más que otra solo por tener mayor matrícula.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

archivo = Path('11_MATRICULAS_ED_SUPERIOR_BIOBIO_2021.xlsx')
if not archivo.exists():
    raise FileNotFoundError('Sube 11_MATRICULAS_ED_SUPERIOR_BIOBIO_2021.xlsx al panel Archivos de Colab.')

df = pd.read_excel(archivo, sheet_name='BASE DE DATOS')
print('Base original:', df.shape)
df.head()

### Variables principales
`GENERO`, `TIPO DE INSTITUCION`, `AREA CONOCIMIENTO`, `JORNADA`, `PROVINCIA SEDE` y `COMUNA SEDE` son cualitativas. `EDAD` y duración son cuantitativas discretas. `VALOR ARANCEL (PESOS)` es cuantitativa y se tratará como continua para el análisis de distribución.

In [ ]:
# Limpieza: pregrado y arancel mayor que cero
pregrado = ['Carreras Profesionales', 'Carreras Tecnicas']
d = df[df['NIVEL CARRERA'].isin(pregrado)].copy()
d = d[d['VALOR ARANCEL (PESOS)'] > 0].copy()
n = len(d)

# Oferta única: combinación disponible que distingue una oferta y su precio
clave_oferta = ['NOMBRE DE INSTITUCION','NOMBRE CARRERA','COMUNA SEDE','MODALIDAD','JORNADA',
                'DURACION TOTAL CARRERA (SEMESTRES)','VALOR MATRICULA (PESOS)','VALOR ARANCEL (PESOS)']
ofertas = d.drop_duplicates(subset=clave_oferta).copy()

print('Matrículas analizadas:', n)
print('Ofertas académicas únicas:', len(ofertas))

## 2. Contexto territorial del Biobío
Antes de responder las preguntas obligatorias, observamos cómo se distribuyen las matrículas y las ofertas entre **Concepción, Biobío y Arauco**. Esto conecta el análisis con una característica interna de la propia región sin inventar causas externas.

In [ ]:
mat_prov = d.groupby('PROVINCIA SEDE').size()
of_prov = ofertas.groupby('PROVINCIA SEDE').size()
med_prov = ofertas.groupby('PROVINCIA SEDE')['VALOR ARANCEL (PESOS)'].median()

tabla_regional = pd.DataFrame({
    'Matrículas': mat_prov,
    '% matrículas': (mat_prov/n*100).round(2),
    'Ofertas únicas': of_prov,
    '% ofertas': (of_prov/len(ofertas)*100).round(2),
    'Arancel mediano oferta ($)': med_prov
}).sort_values('Matrículas', ascending=False)
display(tabla_regional)

ax=(tabla_regional['Arancel mediano oferta ($)']/1_000_000).plot(kind='bar', edgecolor='black', figsize=(8,5))
ax.set_title('Arancel mediano de las ofertas por provincia del Biobío')
ax.set_ylabel('Millones de pesos'); ax.set_xlabel('Provincia'); plt.xticks(rotation=0); plt.show()

**Lectura regional.** Concepción concentra la mayor parte de las matrículas y de las ofertas únicas y presenta un arancel mediano de oferta mayor que Biobío y Arauco. Esto describe una **concentración territorial**, pero no demuestra que la provincia cause el precio, porque la mezcla de carreras e instituciones también cambia entre territorios.

# ÍTEM 1 — Descripción general
Se incluyen frecuencias, tendencia central, percentiles, dispersión y gráficos. En esta sección los cálculos describen principalmente **matrículas**.

In [ ]:
# Frecuencia: área del conocimiento
fa = d['AREA CONOCIMIENTO'].value_counts()
tabla_area = pd.DataFrame({'f':fa,'F':fa.cumsum(),'h %':(fa/n*100).round(2),'H %':(fa/n*100).round(2).cumsum()})
display(tabla_area)

# Arancel agrupado en 8 intervalos
d['Intervalo_Arancel']=pd.cut(d['VALOR ARANCEL (PESOS)'], bins=8, include_lowest=True)
fi=d['Intervalo_Arancel'].value_counts(sort=False)
tabla_arancel=pd.DataFrame({'f':fi,'F':fi.cumsum(),'h %':(fi/n*100).round(2),'H %':(fi/n*100).round(2).cumsum()})
display(tabla_arancel)

# Género x tipo de institución, porcentajes dentro de institución
tabla_genero=pd.crosstab(d['GENERO'],d['TIPO DE INSTITUCION'],normalize='columns')*100
display(tabla_genero.round(2))

In [ ]:
arancel=d['VALOR ARANCEL (PESOS)']; edad=d['EDAD']; dur=d['DURACION TOTAL CARRERA (SEMESTRES)']

tendencia=pd.DataFrame({
 'Media':[arancel.mean(),edad.mean(),dur.mean()],
 'Mediana':[arancel.median(),edad.median(),dur.median()],
 'Moda':[arancel.mode()[0],edad.mode()[0],dur.mode()[0]]
},index=['Arancel','Edad','Duración'])
display(tendencia.round(2))

percentiles=pd.DataFrame({
 'Arancel ($)':arancel.quantile([.10,.25,.50,.75,.90,.95]),
 'Edad':edad.quantile([.10,.25,.50,.75,.90,.95])})
display(percentiles)

dispersion=pd.DataFrame({
 'Rango':[arancel.max()-arancel.min(),edad.max()-edad.min(),dur.max()-dur.min()],
 'Varianza':[arancel.var(),edad.var(),dur.var()],
 'Desv. estándar':[arancel.std(),edad.std(),dur.std()],
 'IQR':[arancel.quantile(.75)-arancel.quantile(.25),edad.quantile(.75)-edad.quantile(.25),dur.quantile(.75)-dur.quantile(.25)]
},index=['Arancel','Edad','Duración'])
display(dispersion.round(2))

In [ ]:
# Gráficos principales del Ítem 1
fig,ax=plt.subplots(figsize=(9,5)); tabla_area['f'].plot(kind='bar',ax=ax,edgecolor='black'); ax.set_title('Matrículas por área del conocimiento'); plt.show()
fig,ax=plt.subplots(figsize=(9,5)); tabla_arancel['f'].plot(kind='bar',ax=ax,edgecolor='black'); ax.set_title('Distribución de matrículas por intervalo de arancel'); plt.xticks(rotation=45,ha='right'); plt.show()
fig,ax=plt.subplots(figsize=(7,7)); d['TIPO DE INSTITUCION'].value_counts().plot(kind='pie',autopct='%1.1f%%',ax=ax); ax.set_ylabel(''); ax.set_title('Matrícula por tipo de institución'); plt.show()

**Interpretación del Ítem 1.** Tecnología y Salud concentran más de la mitad de la matrícula. Los aranceles muestran una distribución con cola hacia valores altos, por lo que la **mediana** y el **IQR** son especialmente útiles. Estas cifras describen matrículas; cuando la pregunta sea sobre cuánto cuesta una carrera típica, se utilizarán ofertas únicas.

# ÍTEM 2 — Pregunta 1
## ¿Hay áreas del conocimiento donde las carreras sean más caras?
**Criterio:** comparar la mediana del arancel de las **ofertas únicas** de cada área con la mediana global de las ofertas. Se agregan Q1, Q3 e IQR para describir la dispersión.

In [ ]:
med_global=ofertas['VALOR ARANCEL (PESOS)'].median()
g=ofertas.groupby('AREA CONOCIMIENTO')['VALOR ARANCEL (PESOS)']
tabla_p1=pd.DataFrame({'Ofertas':g.size(),'Q1':g.quantile(.25),'Mediana':g.median(),'Q3':g.quantile(.75)})
tabla_p1['IQR']=tabla_p1['Q3']-tabla_p1['Q1']
tabla_p1['Diferencia vs global']=tabla_p1['Mediana']-med_global
tabla_p1=tabla_p1.sort_values('Mediana',ascending=False)
print('Mediana global de ofertas:', med_global); display(tabla_p1)

ax=(tabla_p1['Mediana']/1_000_000).plot(kind='bar',figsize=(11,5),edgecolor='black')
ax.axhline(med_global/1_000_000,linestyle='--',label='Mediana global'); ax.legend()
ax.set_title('Arancel mediano de ofertas únicas por área'); ax.set_ylabel('Millones de pesos'); plt.xticks(rotation=45,ha='right'); plt.show()

**Respuesta 1.** Sí. La mediana global de las 1.273 ofertas es cercana a **$2,03 millones**. Destacan **Derecho ($3,586 millones)**, **Ciencias Básicas ($3,290 millones)** y **Agropecuaria ($2,982 millones)**. El contexto territorial muestra además que Concepción concentra la mayor oferta y un arancel mediano superior al de Biobío y Arauco. Esto describe diferencias observadas, no causas.

# Pregunta 2
## ¿Qué influencia tiene la edad en el tipo de institución?
Con datos observacionales hablaremos de **asociación**, no causalidad. La herramienta principal será una tabla de contingencia con porcentajes por rango de edad.

In [ ]:
orden=['15 a 19','20 a 24','25 a 29','30 a 34','35 a 39','40 y mas']
tabla_p2=pd.crosstab(d['RANGO EDAD'].str.strip(),d['TIPO DE INSTITUCION'],normalize='index')*100
tabla_p2=tabla_p2.reindex(orden).round(2)
display(tabla_p2)

graf=tabla_p2.drop(columns=['Universidades (* Carrera en Convenio)'],errors='ignore')
graf.plot(kind='bar',stacked=True,figsize=(11,6),edgecolor='black')
plt.title('Tipo de institución según rango de edad'); plt.ylabel('% dentro del rango'); plt.xlabel('Rango de edad')
plt.legend(title='Institución',bbox_to_anchor=(1.02,1),loc='upper left'); plt.xticks(rotation=0); plt.tight_layout(); plt.show()

# Complemento: jornada
tabla_jornada=(pd.crosstab(d['JORNADA'],d['TIPO DE INSTITUCION'],normalize='columns')*100).round(1)
display(tabla_jornada)

**Respuesta 2.** Sí se observa asociación. Entre **15–19 años**, CRUCH representa aproximadamente **46,24%** e IP **19,30%**. En **40 años o más**, IP sube a aproximadamente **52,85%** y CRUCH baja a **10,90%**. El patrón es gradual: con mayor edad aumenta la participación relativa de IP y CFT. La jornada aporta contexto, pero no se presentará como causa demostrada.

# Pregunta 3
## ¿Hay carreras cuyo arancel sea sustantivamente más caro que la mayoría?
Se usa el criterio **$Q_3 + 1,5\times IQR$** sobre ofertas únicas. Así una oferta con muchos estudiantes no altera artificialmente el umbral.

In [ ]:
a=ofertas['VALOR ARANCEL (PESOS)']; Q1=a.quantile(.25); Q3=a.quantile(.75); IQR=Q3-Q1; limite=Q3+1.5*IQR
altas=ofertas[ofertas['VALOR ARANCEL (PESOS)']>limite].copy()
resumen=pd.DataFrame({'Valor':[Q1,Q3,IQR,limite,len(altas),len(altas)/len(ofertas)*100]},
 index=['Q1','Q3','IQR','Límite','Ofertas sobre límite','% de ofertas'])
display(resumen)
print('Por tipo de institución'); display(altas['TIPO DE INSTITUCION'].value_counts())
print('Por provincia'); display(altas['PROVINCIA SEDE'].value_counts())
print('Por área'); display(altas['AREA CONOCIMIENTO'].value_counts())

# Ranking de carreras con al menos 3 ofertas
r=ofertas.groupby('NOMBRE CARRERA')['VALOR ARANCEL (PESOS)'].agg(['count','median'])
r=r[r['count']>=3].sort_values('median',ascending=False).head(10)
ax=(r['median']/1_000_000).plot(kind='bar',figsize=(12,5),edgecolor='black')
ax.set_title('Carreras con mayor arancel mediano'); ax.set_ylabel('Millones de pesos'); plt.xticks(rotation=45,ha='right'); plt.tight_layout(); plt.show()

**Respuesta 3.** $Q_1\approx$ **$1,616 millones**, $Q_3\approx$ **$2,580 millones**, $IQR\approx$ **$0,964 millones** y el límite queda cerca de **$4,026 millones**. Lo superan **135 ofertas**, aproximadamente **10,6%** de las ofertas únicas. Territorialmente, **129 de 135** están en la provincia de Concepción, 6 en Biobío y ninguna en Arauco. Todas pertenecen a universidades CRUCH o privadas. Esto revela concentración regional, pero no prueba que la ubicación sea la causa del precio.

# Conclusiones
**El precio de elegir en Biobío no se distribuye de manera uniforme.**

1. **Territorio:** Concepción concentra la mayor parte de matrículas y ofertas, además de la gran mayoría de ofertas por sobre el umbral IQR.
2. **Área:** Derecho, Ciencias Básicas y Agropecuaria presentan las mayores medianas entre ofertas únicas.
3. **Edad e institución:** los jóvenes tienen mayor presencia relativa en CRUCH; en edades mayores crecen IP y CFT.
4. **Precios extremos:** 135 ofertas superan el límite de $4,026 millones.

### Aprendizajes metodológicos
- Para estudiantes usamos **matrículas**; para precios usamos **ofertas únicas**.
- La **mediana** y el **IQR** son más robustos frente a valores extremos.
- Asociación no significa causalidad.
- Provincia y comuna permiten relacionar los resultados directamente con la Región del Biobío.

### Limitaciones
La base corresponde a un solo año, registra aranceles de referencia y no pagos efectivos, y no contiene todas las variables socioeconómicas necesarias para explicar las decisiones de matrícula.